# 롱 컨텍스트 처리 - 실습 코드 1: RoPE Scaling 구현: 컨텍스트 윈도우 확장

- Tutorial ID: `expand-long-context`
- Tutorial: 롱 컨텍스트 처리
- Section ID: `expand-long-context-code-1`
- Section: 실습 코드 1: RoPE Scaling 구현: 컨텍스트 윈도우 확장


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 1: RoPE Scaling 구현: 컨텍스트 윈도우 확장
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) 위치 정보가 벡터 회전/편향으로 attention score에 들어가는 방식 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import torch
import torch.nn as nn
import math

# ── Position Interpolation (PI) ──
def position_interpolation_scaling(pos, original_max_len, target_max_len):
    """Position Interpolation: 위치 인덱스를 [0, original_max_len]으로 압축"""
    scale = original_max_len / target_max_len
    return pos * scale

# ── NTK-aware Scaling ──
class ScaledRoPE(nn.Module):
    """NTK-aware RoPE Scaling 구현
    
    컨텍스트 윈도우를 확장하면서도 
    고빈도 위치 정보를 보존하는 방법
    """
        def __init__(self, dim, max_seq_len=8192, base=10000, scaling_factor=4.0):
        super().__init__()
        self.dim = dim
        self.max_seq_len = max_seq_len
        self.scaling_factor = scaling_factor
        
        # NTK-aware base 계산
        # 원래: base = 10000
        # 확장: base = base * scaling_factor^(dim/(dim-2))
        self.base = base * (scaling_factor ** (dim / (dim - 2)))
        
        inv_freq = 1.0 / (self.base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer('inv_freq', inv_freq)
    
        def forward(self, x, seq_len=None):
        """x: (batch, seq, heads, dim)"""
        t = torch.arange(seq_len or x.shape[1], device=x.device, dtype=self.inv_freq.dtype)
        freqs = torch.outer(t, self.inv_freq)  # (seq, dim//2)
        emb = torch.cat([freqs, freqs], dim=-1)  # (seq, dim)
        return emb.cos(), emb.sin()

def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(q, k, cos, sin):
    cos = cos.unsqueeze(0).unsqueeze(2)  # (1, seq, 1, dim)
    sin = sin.unsqueeze(0).unsqueeze(2)
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed

# ── 비교 실험 ──
print("=== RoPE Scaling 비교 ===\n")
dim = 64
original_rope = ScaledRoPE(dim, max_seq_len=2048, scaling_factor=1.0)
scaled_rope = ScaledRoPE(dim, max_seq_len=8192, scaling_factor=4.0)

# 4096번째 위치에서의 임베딩 비교
pos_4096 = torch.zeros(1, 4097, 8, dim)  # 4096번째 위치까지
cos_orig, sin_orig = original_rope(pos_4096, seq_len=4097)
cos_scaled, sin_scaled = scaled_rope(pos_4096, seq_len=4097)

print(f"원래 RoPE (2048 학습): cos[4096] 범위 = [{cos_orig[0, 4096].min():.3f}, {cos_orig[0, 4096].max():.3f}]")
print(f"스케일된 RoPE (8192용): cos[4096] 범위 = [{cos_scaled[0, 4096].min():.3f}, {cos_scaled[0, 4096].max():.3f}]")
print(f"\n→ NTK-aware Scaling은 고빈도 정보를 보존하면서 외삽합니다")